# Research_2DGS: Anisotropic BRDF in 2DGS for Inverse Rendering
This notebook sets up the environment, trains the Anisotropic GGX PBR model on 2D Gaussian Splatting, renders 2x3 combined videos, and tests Relighting under HDR environment maps.

### Step 1: Check GPU and CUDA Toolkit

In [ ]:
!nvidia-smi
!nvcc --version

### Step 2: Clone Code recursively

In [ ]:
!git clone --recursive https://github.com/shInNei/Research_2DGS.git /content/Research_2DGS
%cd /content/Research_2DGS
!git fetch origin
!git checkout main
!git pull origin main

### Step 3: Install dependencies and compile CUDA submodules

In [ ]:
%cd /content/Research_2DGS
!apt-get update -qq && apt-get install -y aria2
!pip install plyfile opencv-python lpips trimesh open3d tqdm mediapy imageio gdown

# Patch simple-knn compiling issue (FLT_MAX not found on modern compilers)
!python -c "with open('submodules/simple-knn/simple_knn.cu', 'r+') as f: c = f.read(); f.seek(0); f.write('#include <cfloat>\n' + c)"

# Compile custom CUDA rasterizer and KNN submodules
!pip install submodules/diff-surfel-rasterization
!pip install submodules/simple-knn

### Step 4: Multi-Thread Parallel Download Dataset (Fast Speed via aria2c)

In [ ]:
%cd /content/Research_2DGS
!mkdir -p data
%cd data

# Fast 16-thread parallel download using aria2c (takes seconds instead of 1 hour)
!aria2c -x 16 -s 16 -k 1M -o lego.zip "https://zenodo.org/records/7880113/files/lego.zip?download=1"
!unzip -q lego.zip -d lego
!rm lego.zip

%cd /content/Research_2DGS

### Step 5: Start Training!

In [ ]:
%cd /content/Research_2DGS
!python train.py -s data/lego --model_path output/tensoir_lego --light_type colocated --eval

### Step 6: Render Trajectory and Evaluate Metrics
Run evaluation to render views, export material maps (Albedo, Normals, Roughness, Metallic), generate combined 2x3 grid video, and compute metrics.

In [ ]:
# Render testing images and video trajectories (includes 2x3 combined video)
!python render.py -m output/tensoir_lego --light_type colocated --render_path --skip_mesh

# Evaluate PSNR, SSIM, LPIPS metrics
!python metrics.py -m output/tensoir_lego

### Step 7: View Metrics and Combined 2x3 Material Video
Display computed metrics and visualize the single combined 2x3 grid trajectory video.

In [ ]:
import json
import os
import mediapy as media

# 1. Print metrics table
metrics_file = "output/tensoir_lego/results.json"
if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        data = json.load(f)
    print("=" * 40)
    print("       Evaluation Metrics (ours_30000)   ")
    print("=" * 40)
    for method, metrics in data.items():
        print(f"Method: {method}")
        for k, v in metrics.items():
            print(f"  {k:<10}: {v:.5f}")
    print("=" * 40)
else:
    print("Metrics file not found.")

# 2. Display Combined 2x3 Grid Video
combined_video = "output/tensoir_lego/traj/ours_30000/render_traj_combined_grid.mp4"
if os.path.exists(combined_video):
    print("\nDisplaying COMBINED 2x3 Grid Video (Color | Albedo | Normal // Roughness | Metallic | Depth):")
    media.show_video(combined_video, height=450)
else:
    print(f"Video {combined_video} not found.")

### Step 8: Backup Trained Model Zip to Google Drive or Local Download

In [ ]:
# Compress trained model folder to zip
%cd /content/Research_2DGS
!zip -r tensoir_lego_checkpoint.zip output/tensoir_lego
print("Compressed output/tensoir_lego to tensoir_lego_checkpoint.zip!")

### Step 9: Restore Checkpoint & Download Official TensoIR Environment Maps Package

In [ ]:
import os
from google.colab import files

# 1. Upload tensoir_lego_checkpoint.zip if output folder does not exist
if not os.path.exists('/content/Research_2DGS/output/tensoir_lego'):
    print('Please upload your tensoir_lego_checkpoint.zip file...')
    uploaded = files.upload()
    !unzip -o -q tensoir_lego_checkpoint.zip -d /content/Research_2DGS/
    print('Checkpoint restored successfully!')
else:
    print('Checkpoint already exists at output/tensoir_lego!')

# 2. Download official TensoIR Environment Maps (high_res_envmaps_1k) from official Google Drive link
%cd /content/Research_2DGS
!mkdir -p data/eval_lights
!gdown --id 10WLc4zk2idf4xGb6nPL43OXTTHvAXSR3 -O /content/Research_2DGS/data/high_res_envmaps_1k.zip
!unzip -o -q /content/Research_2DGS/data/high_res_envmaps_1k.zip -d /content/Research_2DGS/data/
!cp /content/Research_2DGS/data/high_res_envmaps_1k/*.hdr /content/Research_2DGS/data/eval_lights/
print('Official TensoIR Environment Maps extracted to data/eval_lights/!')

### Step 10: Test Relighting under Official TensoIR city.hdr Map

In [ ]:
%cd /content/Research_2DGS

# 1. Render Relighting under official TensoIR city.hdr
!python render_relight.py -m output/tensoir_lego --hdr_path data/eval_lights/city.hdr

# 2. Measure Relighting Metrics against GT
!python metrics.py -m output/tensoir_lego/relight/hdr_city